# 07 — Run 30 Paired Replications

## Thiết kế thống kê hiện tại

Với sample hiện có:
- 3 scenarios: `normal`, `peak`, `high_demand`;
- 30 independent replications/scenario;
- 4 policies: B0, B3, B4, B6.

Số **exogenous realizations**:
\[
3\times30=90
\]

Số policy runs:
\[
3\times30\times4=360
\]

Mỗi `(scenario_id, replication_id)` phải dùng **cùng random seed** cho B0/B3/B4/B6.

So sánh thống kê dùng **30 paired replication differences trên từng scenario**, không coi 9.000 incidents là 9.000 independent experiments.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, cwd.parent, *cwd.parents] if (p/"data"/"sample").exists()), None)
if ROOT is None: raise FileNotFoundError("Không tìm thấy project root.")

reps = pd.read_csv(ROOT/"data"/"sample"/"replications.csv.gz")
POLICIES = ["B0","B3","B4","B6"]

manifest = reps[[
    "scenario_id","replication_id","random_seed","common_random_group"
]].assign(_key=1).merge(
    pd.DataFrame({"policy_id":POLICIES,"_key":1}),
    on="_key"
).drop(columns="_key")

display(manifest.head(12))
print("Exogenous realizations:", len(reps))
print("Policy runs:", len(manifest))

## 1. Paired-seed audit

In [ ]:
audit = (
    manifest.groupby(["scenario_id","replication_id"])
    .agg(
        policy_count=("policy_id","nunique"),
        seed_count=("random_seed","nunique"),
        common_group_count=("common_random_group","nunique"),
    )
    .reset_index()
)

audit["paired_ok"] = (
    (audit["policy_count"] == len(POLICIES))
    & (audit["seed_count"] == 1)
    & (audit["common_group_count"] == 1)
)

display(audit.head())
print("All paired:", bool(audit["paired_ok"].all()))

## 2. Expected run counts

In [ ]:
expected = (
    manifest.groupby("scenario_id")
    .agg(
        replications=("replication_id","nunique"),
        policy_runs=("policy_id","count")
    )
)
expected

Kỳ vọng mỗi scenario:
- 30 replications;
- 120 policy runs.

Toàn bộ:
- 90 exogenous realizations;
- 360 policy runs.

## 3. Run manifest export

In [ ]:
OUT = ROOT/"experiments"/"paired_30"
OUT.mkdir(parents=True, exist_ok=True)

manifest.to_csv(OUT/"run_manifest.csv", index=False)
audit.to_csv(OUT/"pairing_audit.csv", index=False)

print("Saved:", OUT/"run_manifest.csv")
print("Saved:", OUT/"pairing_audit.csv")

## 4. Required per-replication metrics

Ít nhất:
- `feasible_rate_overall`
- `feasible_rate_P1/P2/P3/P4`
- `SLA_pass_rate`
- `SOC_violation_count`
- `hospital_capacity_violations`
- `assignments_to_charging`
- `avg_response_time_min`

Nghiệm chính nên bổ sung:
- `P95_response_P1`
- `time_to_compatible_hospital`
- `solver_time_sec`
- `mip_gap`

In [ ]:
REQUIRED_METRICS = [
    "feasible_rate_overall",
    "feasible_rate_P1","feasible_rate_P2","feasible_rate_P3","feasible_rate_P4",
    "SLA_pass_rate",
    "SOC_violation_count",
    "hospital_capacity_violations",
    "assignments_to_charging",
    "avg_response_time_min",
    "P95_response_P1",
    "time_to_compatible_hospital",
]
pd.DataFrame({"metric":REQUIRED_METRICS})

## 5. Paired comparison gate

Không chạy paired statistics nếu chưa có đủ:
```text
scenario × replication × policy
```
cho cả 4 policies.

Mỗi scenario cần **30 paired differences** cho từng comparison, ví dụ:
- B6 − B0
- B6 − B3
- B6 − B4

Primary reporting nên dùng:
- mean/median paired difference;
- 95% CI trên vector 30 paired differences;
- effect size;
- không bootstrap 100 incident rows rồi gọi đó là 100 independent replications.

In [ ]:
def pairing_completeness(metrics: pd.DataFrame):
    counts = (
        metrics.groupby(["scenario_id","replication_id"])["policy_id"]
        .nunique()
        .reset_index(name="policy_count")
    )
    return {
        "rows": len(metrics),
        "pairs_expected": 90,
        "pairs_present": int((counts["policy_count"] == 4).sum()),
        "complete": bool(len(counts)==90 and (counts["policy_count"]==4).all()),
    }

print("Function ready; chờ replication_metrics từ DES runner.")

# Stop condition

Đến đây bạn đã chuẩn bị đúng **protocol cho 30 paired replications**.

Nhưng **không được chạy/diễn giải paired result cuối cùng** cho tới khi:
1. Notebook 02 OSMnx QA pass;
2. Notebook 03 traffic provenance rõ;
3. Notebook 04 energy/hard-feasibility audit pass;
4. Notebook 05 rolling DES chạy ổn định;
5. Notebook 06 B6 MILP thật sự chạy và log solver diagnostics;
6. mỗi scenario có đủ 30 paired replications cho cả B0/B3/B4/B6.